![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## EODHD Upcoming IPOs Research

This notebook measures IPO first-day returns, calculated as the change from the IPO day's opening price to the next trading day's opening price.

In [ ]:
qb = QuantBook()
# Anchor the research clock to the start of 2026.
qb.set_start_date(2026, 1, 1)
# Daily bars will have an end_time that matches the following midnight.
qb.settings.daily_precise_end_time = False
# Add the data feed.
ipo_data = qb.add_data(EODHDUpcomingIPOs, "IPOS")

### Build a Upcoming IPOs Universe

Select confirmed non-penny upcoming IPOs, then inspect the returned universe history.

In [8]:
# Pull the dataset history.
universe_history = qb.history(EODHDUpcomingIPOs, qb.time - timedelta(365), qb.time)
# Define target price columns.
price_cols = ['lowestprice', 'highestprice', 'offerprice']
# Filter EODHDUpcomingIPOs dataframe.
universe_history = universe_history[
    universe_history['ipodate'].notna() &
    universe_history['dealtype'].isin([EODHD.DealType.EXPECTED, EODHD.DealType.PRICED]) &
    (universe_history[price_cols].min(axis=1) > 1)
]
# Print the returned shape and columns.
print(f"Shape: {universe_history.shape}")
print(f"Columns: {list(universe_history.columns)}")
universe_history.head()

In [ ]:
# Take the first row per listing to read its IPO date.
ipo_events = universe_history.groupby(level=0).first()
print(f"IPO listings: {len(ipo_events)}")
ipo_events[['ipodate', 'name']].head()

### Universe Diagnostics

Check how many assets pass the filter each day and summarize the factors.

In [ ]:
# Count how many symbols pass the filter each day.
universe_size = universe_history.groupby(level=1).size()
print(f"Universe days: {len(universe_size)}")
# Keep the unique list of selected symbols.
unique_assets = list(universe_history.index.unique(0))
print(f"Mean universe size per day: {universe_size.mean():.1f}")
universe_size.plot(title='Daily Universe Size', ylabel='Universe Size');

### Daily Universe Prices

Fetch daily price history for every symbol that appears in the universe.

In [11]:
# Fetch daily price history for every listing. 
symbols = list(ipo_events.index)
# Start one day before the earliest listing.
start_time = ipo_events['ipodate'].min() - timedelta(1)
history = qb.history(symbols, start_time, qb.time, Resolution.DAILY)
history

### IPO First-Day Returns

Take each listing's open-to-open return across its IPO date, one row per IPO.

In [ ]:
# Order each listing's bars and take the first two opens.
opens = history['open'].groupby(level=0)
# The first traded open is the IPO date open; the next is the following day.
ipo_events['ipo_open'] = opens.nth(0).droplevel(1)
ipo_events['next_open'] = opens.nth(1).droplevel(1)
# First-day return is the open-to-open move across the IPO date.
ipo_events['first_day_return'] = (ipo_events['next_open'] - ipo_events['ipo_open']) / ipo_events['ipo_open']
returns = ipo_events['first_day_return'].dropna()
print(f"Sample Size: {len(returns)}")
print(f"Mean: {returns.mean() * 100:.2f}%")
print(f"Median: {returns.median() * 100:.2f}%")
print(f"Max: {returns.max() * 100:.2f}%")
print(f"Min: {returns.min() * 100:.2f}%")
ipo_events[['ipodate', 'ipo_open', 'next_open', 'first_day_return']].dropna().head()